# Energies and Kinetic Properties

In this notebook, we explore various properties of the particle system under investigation using our molecular dynamics (MD) simulation.

---

### Outline:
1. **Potential Energy**: Average potential energy of all particles.
2. **Kinetic Properties**: Dynamic system properties based on the average velocities of all particles:
    - **Average Velocities** ($V_x, V_y, V_\phi (\omega)$)
    - **Temperature** ($T_{\text{internal}}, T_{\text{real}}$)
    - **Kinetic Energies** ($K_x, K_y, K_\phi$)
    - **Center of Mass Velocity** (linear and rotational)
3. **Total Energy**: Conservation of energy in the system.

---
Loading file ...

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

path1 = 'dec18_dotSingleRandomVv'
path2 = 'dec21_dotSingleRandomVv'

---

## 1. Potential Energy

The **potential energy of the system** depends on the particle type and the number of **degrees of freedom** (e.g., `particleDot` and `particleOriented`). Interaction energy between particle pairs determines the energy experienced by each particle in the system. This energy evolves during the simulation, influenced by the **configuration** and **positions** of particles within the simulation domain.

The plot below illustrates the trend of the **average** potential energy experienced by each particle due to interactions with other particles over the simulation period. This data is derived from the output of the [MD simulation code](/home/hadis/custom_vector/test/MDsimulation/nParticleMD.cpp), which computes the potential energy for each particle at every simulation step. The average is calculated in a Python script. The calculation leverages the particle interaction potential energy, which is further analyzed in the [Potential and Force notebook](/home/hadis/custom_vector/notebooks/MD_Simulation/01_PotentialForceFormula.ipynb).

For `particleDot`, the Lennard-Jones potential is used, defined as:
$$
U_{LJ}(r) = 4\epsilon \left[ \left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6} \right]
$$

For `particleOriented`, a modified version of the Lennard-Jones potential is employed:
$$
U_{\text{mod}}(r, \Delta \phi) = U_{LJ}(r) + A(r) \cos(n \Delta \phi)
$$


In [2]:
def load_potential_energy(path):
    """Load and process potential energy data from a given path."""
    file_path = f"/home/hadis/custom_vector/buildParticleOriented/buildVS/{path}/N_particle_PotentialEnergyMD.dat"
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    data = np.loadtxt(file_path)
    return np.sum(data, axis=1) / data.shape[1]

def plot_potential(paths):
    """Plot potential energy evolution."""
    if isinstance(paths, str):
        paths = [paths]
    
    fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
    
    for path in paths:
        POTE = load_potential_energy(path)
        ax.plot(POTE, label=f"{path} $U$")
    
    ax.set_xlabel("Time Steps", fontsize=12)
    ax.set_ylabel("Potential Energy", fontsize=12)
    ax.set_title("Potential Energy Evolution", fontsize=14)
    ax.legend()
    ax.grid(True)
    
    plt.tight_layout()
    plt.show()


In [ ]:
plot_potential([path2])


---

## 2. Kinetic Properties

### 2.1. Average Velocities ($V_x, V_y, V_\phi (\omega)$)

Technically speaking, the **velocities** play a crucial role in determining the **kinetic energy** and **temperature** of the system. However, during the investigation of various properties, there might be **inconsistencies** due to either bugs or minor issues in the implementation. Therefore, it is helpful to **double-check** and compare related parameters to ensure accuracy.  

The first property in the **Kinetic Properties** category is the **velocity**. To examine the behavior of particle velocities, we output the velocity of each particle for different components (**x**, **y**, and **phi**, if applicable) in the main code, [MD simulation](/home/hadis/custom_vector/test/MDsimulation/nParticleMD.cpp).

Below, we present the **average velocities** for various components and compare their values and trends with other properties of the system:


---

### 2.2. Temperature Calculations ($T_{raw}, T_{actual}$)

To calculate the **temperature** of the system as a macroscopic property, we use the **equipartition theorem** from statistical mechanics. The theorem states that each degree of freedom contributes equally to the total kinetic energy of the system, and the average kinetic energy per degree of freedom is proportional to the temperature.

Accurate temperature measurement is essential for analyzing the system and for controlling its thermal state using thermostats. However, due to the complex movements of particles in molecular dynamics (MD), we must account for artificial effects. This is achieved by distinguishing between two components:
- **Raw Temperature** ($T_{raw}$): Based on uncorrected particle velocities.
- **Actual Temperature** ($T_{actual}$): Corrected for artificial effects like **flying ice cube** and **rotating ice cube** motions.  

In the following sections, we explain the physics behind these calculations (implemented in [MD simulation code](/home/hadis/custom_vector/test/MDsimulation/nParticleMD.cpp)) and visualize the simulation outputs.

---

#### Degrees of Freedom ($f$)

The relationship between kinetic energy and temperature is given by:
$$
\frac{f}{2} N k_B T = K_{\text{total}},
$$
where:
- $f$ is the number of degrees of freedom per particle,
- $N$ is the number of particles,
- $k_B$ is the Boltzmann constant,
- $K_{\text{total}}$ is the total kinetic energy of the system.

For example:
- In a system with `particleDot` (translational motion in $x$ and $y$), $f = 2$.
- For `particleOriented` (translational motion in $x, y$ and rotational motion $\phi$), $f = 3$.

---

#### 1. Raw Temperature ($T_{raw}$)

The raw temperature is calculated using the uncorrected velocities of particles. For different particle types, the total kinetic energy is:

- For `particleDot`:
  $$
  K_{\text{total}} = \sum_{i=1}^N \frac{1}{2} m v_i^2,
  $$
  where $v_i$ is the velocity of particle $i$ and $m$ is its mass.

- For `particleOriented`:
  $$
  K_{\text{total}} = \sum_{i=1}^N \frac{1}{2} \left( m v_i^2 + I \omega_i^2 \right),
  $$
  where $\omega_i$ is the angular velocity of particle $i$, and $I$ is its moment of inertia.

Using this kinetic energy, the raw temperature is:
$$
T_{raw} = \frac{2}{f} \frac{K_{\text{total}}}{N k_B}.
$$

---

#### 2. Actual Temperature ($T_{actual}$)

To calculate the actual temperature, corrections are applied to eliminate artificial effects:

- **Flying Ice Cube Correction:**
  Subtract the center of mass (COM) velocity from each particle's velocity:
  $$
  \mathbf{v'}_i = \mathbf{v}_i - \mathbf{v}_{COM},
  $$
  where:
  $$
  \mathbf{v}_{COM} = \frac{1}{N} \sum_{i=1}^N \mathbf{v}_i.
  $$

- **Rotating Ice Cube Correction:**
  - Calculate the system's angular velocity:
    $$
    \boldsymbol{\omega}_{sys} = \frac{\sum_{i=1}^N \mathbf{r}_i \times m \mathbf{v}_i}{\sum_{i=1}^N m |\mathbf{r}_i|^2},
    $$
    where $\mathbf{r}_i = \mathbf{x}_i - \mathbf{x}_{COM}$ is the position of particle $i$ relative to the center of mass.
  - Subtract the rotational velocity component:
    $$
    \mathbf{v}_i^{final} = \mathbf{v'}_i - (\boldsymbol{\omega}_{sys} \times \mathbf{r}_i).
    $$

- **Recalculate Total Kinetic Energy:**
  After applying these corrections:
  - For `particleDot`:
    $$
    K'_{\text{total}} = \sum_{i=1}^N \frac{1}{2} m |\mathbf{v}_i^{final}|^2.
    $$
  - For `particleOriented`:
    $$
    K'_{\text{total}} = \sum_{i=1}^N \frac{1}{2} m |\mathbf{v}_i^{final}|^2 + \sum_{i=1}^N \frac{1}{2} I \omega_i^2.
    $$

- **Compute Actual Temperature:**
  Substitute the corrected kinetic energy into the temperature equation:
  $$
  T_{actual} = \frac{2}{f} \frac{K'_{\text{total}}}{N k_B}.
  $$

---

#### Final Outcome:
- **Raw Temperature ($T_{raw}$):** Calculated using uncorrected velocities ($\mathbf{v}_i$).
- **Actual Temperature ($T_{actual}$):** Calculated using fully corrected velocities ($\mathbf{v}_i - \mathbf{v}_{COM} - \boldsymbol{\omega}_{sys} \times \mathbf{r}_i$).

These corrections ensure that the actual temperature reflects the intrinsic thermal energy of the system, free from artificial translational and rotational artifacts.

---

#### Visualization:
Below is a plot showing the temperature evolution over time during the MD simulation, based on the output of [MD simulation code](/home/hadis/custom_vector/test/MDsimulation/nParticleMD.cpp).
